In [ ]:
# optimizers whose actual update the weight parameters of the model, and the loss function which calculates the error of the model.

import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()


# ============================================================
# DENSE LAYER
# ============================================================

class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases  = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        # gradient on weights = inputs.T × incoming gradient
        # shape: (n_inputs, batch) × (batch, n_neurons) = (n_inputs, n_neurons)
        self.dweights = np.dot(self.inputs.T, dvalues)

        # gradient on biases = sum across batch dimension
        # keepdims → stays (1, n_neurons) matching bias shape
        self.dbiases  = np.sum(dvalues, axis=0, keepdims=True)

        # gradient on inputs = incoming gradient × weights.T
        # needed to pass gradient back to previous layer
        self.dinputs  = np.dot(dvalues, self.weights.T)


# ============================================================
# ACTIVATIONS
# ============================================================

class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        # where input was negative → gradient = 0 (blocked)
        # where input was positive → gradient passes through unchanged
        self.dinputs[self.inputs <= 0] = 0


class Activation_Softmax:
    def forward(self, inputs):
        self.inputs  = inputs
        exp_values   = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output  = probabilities

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)
        for index, (single_output, single_dvalues) in \
                enumerate(zip(self.output, dvalues)):
            single_output    = single_output.reshape(-1, 1)
            jacobian_matrix  = np.diagflat(single_output) - \
                               np.dot(single_output, single_output.T)
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)


# ============================================================
# LOSS
# ============================================================

class Loss:
    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        return np.mean(sample_losses)


class Loss_CategoricalCrossentropy(Loss):
    def forward(self, y_pred, y_true):
        samples         = len(y_pred)
        y_pred_clipped  = np.clip(y_pred, 1e-7, 1 - 1e-7)

        if len(y_true.shape) == 1:        # sparse labels
            correct_confidences = y_pred_clipped[range(samples), y_true]
        elif len(y_true.shape) == 2:      # one-hot labels
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)

        return -np.log(correct_confidences)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        labels  = len(dvalues[0])

        # convert sparse to one-hot if needed
        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]

        # gradient of cross entropy = -y_true / y_pred
        self.dinputs = -y_true / dvalues
        # normalize by batch size so lr isn't batch-size dependent
        self.dinputs = self.dinputs / samples


# combined softmax + cross entropy backward (faster + more stable)
class Activation_Softmax_Loss_CategoricalCrossentropy:
    def backward(self, dvalues, y_true):
        samples = len(dvalues)

        if len(y_true.shape) == 2:        # one-hot → sparse
            y_true = np.argmax(y_true, axis=1)

        self.dinputs = dvalues.copy()
        # subtract 1 from predicted probability of correct class
        # this is the simplified combined gradient
        self.dinputs[range(samples), y_true] -= 1
        # normalize
        self.dinputs = self.dinputs / samples


# ============================================================
# OPTIMIZER 1 — SGD + MOMENTUM + DECAY
# ============================================================

class Optimizer_SGD:
    def __init__(self, learning_rate=1.0, decay=0.0, momentum=0.0):
        self.learning_rate         = learning_rate
        self.current_learning_rate = learning_rate  # changes over time with decay
        self.decay                 = decay
        self.iterations            = 0
        self.momentum              = momentum

    def pre_update_params(self):
        if self.decay:
            # reduce lr each iteration → fine-tune near minimum
            self.current_learning_rate = self.learning_rate * \
                (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if self.momentum:
            # first time → create zero momentum arrays for this layer
            if not hasattr(layer, 'weight_momentums'):
                layer.weight_momentums = np.zeros_like(layer.weights)
                layer.bias_momentums   = np.zeros_like(layer.biases)

            # momentum: keep 90% of old direction + add new gradient direction
            weight_updates = self.momentum * layer.weight_momentums \
                           - self.current_learning_rate * layer.dweights
            layer.weight_momentums = weight_updates  # save for next iteration

            bias_updates = self.momentum * layer.bias_momentums \
                         - self.current_learning_rate * layer.dbiases
            layer.bias_momentums = bias_updates

        else:
            # vanilla SGD — no memory of past
            weight_updates = -self.current_learning_rate * layer.dweights
            bias_updates   = -self.current_learning_rate * layer.dbiases

        layer.weights += weight_updates
        layer.biases  += bias_updates

    def post_update_params(self):
        self.iterations += 1  # increment so decay works next round


# ============================================================
# OPTIMIZER 2 — ADAGRAD
# ============================================================

class Optimizer_Adagrad:
    def __init__(self, learning_rate=1.0, decay=0.0, epsilon=1e-7):
        self.learning_rate         = learning_rate
        self.current_learning_rate = learning_rate
        self.decay                 = decay
        self.iterations            = 0
        self.epsilon               = epsilon  # prevents division by zero

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * \
                (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache   = np.zeros_like(layer.biases)

        # accumulate squared gradients — grows forever (AdaGrad weakness)
        layer.weight_cache += layer.dweights ** 2
        layer.bias_cache   += layer.dbiases  ** 2

        # divide lr by √cache → big past gradients = smaller future steps
        layer.weights += -self.current_learning_rate * layer.dweights / \
                         (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases  += -self.current_learning_rate * layer.dbiases  / \
                         (np.sqrt(layer.bias_cache)   + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


# ============================================================
# OPTIMIZER 3 — RMSPROP
# ============================================================

class Optimizer_RMSprop:
    def __init__(self, learning_rate=0.001, decay=0.0,
                 epsilon=1e-7, rho=0.9):
        self.learning_rate         = learning_rate
        self.current_learning_rate = learning_rate
        self.decay                 = decay
        self.iterations            = 0
        self.epsilon               = epsilon
        self.rho                   = rho  # how much old cache to keep (0.9 = 90%)

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * \
                (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache   = np.zeros_like(layer.biases)

        # moving average — old cache fades, new gradient added
        # fixes AdaGrad's dying lr problem
        layer.weight_cache = self.rho * layer.weight_cache + \
                             (1 - self.rho) * layer.dweights ** 2
        layer.bias_cache   = self.rho * layer.bias_cache   + \
                             (1 - self.rho) * layer.dbiases  ** 2

        layer.weights += -self.current_learning_rate * layer.dweights / \
                         (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases  += -self.current_learning_rate * layer.dbiases  / \
                         (np.sqrt(layer.bias_cache)   + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


# ============================================================
# OPTIMIZER 4 — ADAM (MOMENTUM + RMSPROP + BIAS CORRECTION)
# ============================================================

class Optimizer_Adam:
    def __init__(self, learning_rate=0.001, decay=0.0,
                 epsilon=1e-7, beta_1=0.9, beta_2=0.999):
        self.learning_rate         = learning_rate
        self.current_learning_rate = learning_rate
        self.decay                 = decay
        self.iterations            = 0
        self.epsilon               = epsilon
        self.beta_1                = beta_1  # momentum decay (1st moment)
        self.beta_2                = beta_2  # cache decay   (2nd moment)

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * \
                (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)  # 1st moment
            layer.weight_cache     = np.zeros_like(layer.weights)  # 2nd moment
            layer.bias_momentums   = np.zeros_like(layer.biases)
            layer.bias_cache       = np.zeros_like(layer.biases)

        # 1st moment — momentum (smooth gradient direction)
        layer.weight_momentums = self.beta_1 * layer.weight_momentums + \
                                 (1 - self.beta_1) * layer.dweights
        layer.bias_momentums   = self.beta_1 * layer.bias_momentums   + \
                                 (1 - self.beta_1) * layer.dbiases

        # bias correction for momentum — early iterations are biased toward 0
        # dividing corrects this → proper scale from step 1
        weight_momentums_corrected = layer.weight_momentums / \
            (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected   = layer.bias_momentums   / \
            (1 - self.beta_1 ** (self.iterations + 1))

        # 2nd moment — RMSprop (adapt lr per weight)
        layer.weight_cache = self.beta_2 * layer.weight_cache + \
                             (1 - self.beta_2) * layer.dweights ** 2
        layer.bias_cache   = self.beta_2 * layer.bias_cache   + \
                             (1 - self.beta_2) * layer.dbiases  ** 2

        # bias correction for cache
        weight_cache_corrected = layer.weight_cache / \
            (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected   = layer.bias_cache   / \
            (1 - self.beta_2 ** (self.iterations + 1))

        # final update: momentum direction / adaptive lr scale
        layer.weights += -self.current_learning_rate * \
                          weight_momentums_corrected / \
                         (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases  += -self.current_learning_rate * \
                          bias_momentums_corrected   / \
                         (np.sqrt(bias_cache_corrected)   + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


# ============================================================
# FULL TRAINING LOOP — EVERYTHING CONNECTED
# ============================================================

# data
X, y = spiral_data(samples=100, classes=3)

# network
dense1  = Layer_Dense(2, 64)
relu1   = Activation_ReLU()
dense2  = Layer_Dense(64, 3)

# combined softmax+loss (faster backward)
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# pick your optimizer — swap freely to compare
# optimizer = Optimizer_Adam(learning_rate=0.05, decay=5e-7)
optimizer = Optimizer_SGD(learning_rate=1.0, decay=1e-3, momentum=0.9)
# optimizer = Optimizer_RMSprop(learning_rate=0.02, decay=1e-5, rho=0.999)

# training loop
for epoch in range(10001):

    # ---- FORWARD PASS ----
    dense1.forward(X)               # raw data → hidden layer
    relu1.forward(dense1.output)    # apply non-linearity
    dense2.forward(relu1.output)    # hidden → output layer

    # calculate loss (softmax applied inside)
    loss = loss_activation.calculate(dense2.output, y)

    # predictions → accuracy
    predictions = np.argmax(loss_activation.output, axis=1)
    if len(y.shape) == 2:
        y_true = np.argmax(y, axis=1)
    else:
        y_true = y
    accuracy = np.mean(predictions == y_true)

    if not epoch % 1000:
        print(f'epoch: {epoch:5d} | '
              f'loss: {loss:.4f} | '
              f'acc: {accuracy:.4f} | '
              f'lr: {optimizer.current_learning_rate:.6f}')

    # ---- BACKWARD PASS ----
    loss_activation.backward(loss_activation.output, y)  # loss+softmax gradient
    dense2.backward(loss_activation.dinputs)              # output layer gradient
    relu1.backward(dense2.dinputs)                        # activation gradient
    dense1.backward(relu1.dinputs)                        # hidden layer gradient

    # ---- UPDATE WEIGHTS ----
    optimizer.pre_update_params()       # apply lr decay
    optimizer.update_params(dense1)     # update hidden layer
    optimizer.update_params(dense2)     # update output layer
    optimizer.post_update_params()      # increment iteration counter